In [ ]:
import glob
import os
import pandas as pd
import numpy as np
import sys

#check Python version
if sys.version_info < (3, 8):
    raise RuntimeError("Python 3.8 or higher required!")
    
#optional: high-quality figure output
#%config InlineBackend.figure_format = 'svg'
#%config InlineBackend.rc = {'figure.figsize': (8, 5)}

print("Ready to analyze!")

#### I. PREPROCESSING

In [ ]:
#specify folder with data
datafolder = r"C"

#define experiment name or name of the folder is experiment name
#(leave empty to auto-generate from folder name)
experiment_name = r""

from preprocessing import get_folder_name
#validate path
base_path = datafolder.rsplit('/', 1)[0]
if not os.path.exists(base_path):
    raise FileNotFoundError(f"directory not found: {base_path}")

#preview
preview_files = glob.glob(datafolder)
n_files = len(preview_files)

if n_files == 0:
    raise FileNotFoundError(f"no .asc files found in: {base_path}")

# Set experiment name
experiment_name = experiment_name or get_folder_name(datafolder)

print(f"Experiment: {experiment_name}")
print(f"Location: {base_path}")
print(f"Files found: {n_files} .asc files")

In [ ]:
#choose between method of catching datafiles
#datafiles = glob.glob(os.path.join(datafolder, '*'))
datafiles = glob.glob(datafolder)

#excluding the *_average.asc-files [for ALV-Input-data]
filtered_files = [f for f in datafiles 
                  if "averaged" not in os.path.basename(f).lower()]

n_excluded = len(datafiles) - len(filtered_files)
print(f"collected {len(filtered_files)} files" + 
      (f" (excluded {n_excluded} averaged)" if n_excluded > 0 else ""))

In [ ]:
#base-data extraction
#extract metadata: angle, temperature, wavelength, refractive index, viscosity
#calculate q and q^2
from preprocessing import extract_data

print("Extracting metadata from files...")

all_data = []
n_failed = 0

for i, file in enumerate(filtered_files, 1):
    extracted_data = extract_data(file)
    if extracted_data is not None:
        filename = os.path.basename(file)
        extracted_data['filename'] = filename  
        all_data.append(extracted_data)
    else:
        n_failed += 1
print()

#create dataframe
if not all_data:
    raise ValueError("NO data extracted!")

df_basedata = pd.concat(all_data, ignore_index=True)
df_basedata.index = df_basedata.index + 1  # 1-based indexing

# Calculate scattering vector q and q²
# q = (4πn/λ) * sin(θ/2), where n=refractive index, λ=wavelength, θ=angle
df_basedata['q'] = abs(
    (4 * np.pi * df_basedata['refractive_index']) / 
    df_basedata['wavelength [nm]'] * 
    np.sin(np.radians(df_basedata['angle [°]']) / 2)
)
df_basedata['q^2'] = df_basedata['q'] ** 2

print(f"successfully extracted: {len(all_data)} files")
if n_failed > 0:
    print(f"failed to extract: {n_failed} files")

In [ ]:
#consistency-check: checks extracted metadata for physical plausibility and internal consistency across all measurement files.
#precomputes c = kT/(6*pi*eta),
from scipy.constants import k as k_B

df_basedata_mod = df_basedata.copy()

warnings_issued = 0

#instrument parameters: wavelength and refractive index hese should be identical across all files
unique_wavelengths = df_basedata_mod['wavelength [nm]'].unique()
unique_ri = df_basedata_mod['refractive_index'].unique()

if len(unique_wavelengths) > 1:
    print(f"[!] multiple wavelengths detected: {unique_wavelengths} nm")
    warnings_issued += 1
else:
    print(f"[ok] wavelength:        {unique_wavelengths[0]:.1f} nm (uniform)")

if len(unique_ri) > 1:
    print(f"[!] multiple refractive indices detected: {unique_ri}")
    warnings_issued += 1
else:
    print(f"[ok] refractive index:  {unique_ri[0]:.4f} (uniform)")

#temperature: small spread expected for a single experiment
mean_T   = df_basedata_mod['temperature [K]'].mean()
std_T    = df_basedata_mod['temperature [K]'].std()
sem_T    = df_basedata_mod['temperature [K]'].sem()
range_T  = df_basedata_mod['temperature [K]'].max() - df_basedata_mod['temperature [K]'].min()

print(f"\n  temperature:        {mean_T:.2f} ± {std_T:.2f} K  "
      f"(range: {range_T:.2f} K, n={len(df_basedata_mod)})")
if range_T > 1.0:
    print(f"[!] temperature range > 1 K.")
    warnings_issued += 1
else:
    print(f"[ok] temperature spread < 1 K.")

#viscosity: should be consistent if T is consistent
mean_eta  = df_basedata_mod['viscosity [cp]'].mean()
std_eta   = df_basedata_mod['viscosity [cp]'].std()
sem_eta   = df_basedata_mod['viscosity [cp]'].sem()
range_eta = df_basedata_mod['viscosity [cp]'].max() - df_basedata_mod['viscosity [cp]'].min()

print(f"\n  viscosity:          {mean_eta:.4f} ± {std_eta:.4f} cP  "
      f"(range: {range_eta:.4f} cP)")
if std_eta / mean_eta > 0.01:  # >1% relative spread
    print(f"[!] viscosity spread > 1% relative.")
    warnings_issued += 1
else:
    print(f"[ok] viscosity spread < 1% relative.")

#angle coverage
n_angles = df_basedata_mod['angle [°]'].nunique()
angle_list = sorted(df_basedata_mod['angle [°]'].unique())
print(f"\n  angles measured:    {angle_list}")
if n_angles < 5:
    print(f"[!] only {n_angles} angles available.")
    warnings_issued += 1
else:
    print(f"[ok] {n_angles} angles — sufficient angles available.")

#q^2 range
q2_min = df_basedata_mod['q^2'].min()
q2_max = df_basedata_mod['q^2'].max()
print(f"\n  q² range:           {q2_min:.4e} – {q2_max:.4e} nm⁻²")

#summary
print("\n" + "=" * 55)
if warnings_issued == 0:
    print("data look consistent.")
else:
    print(f"  {warnings_issued} warning(s) issued — review before proceeding.")

#summary dataframe
df_basedata_stats = pd.DataFrame({
    'mean temperature [K]':  [mean_T],
    'std temperature [K]':   [std_T],
    'sem temperature [K]':   [sem_T],
    'mean viscosity [cp]':   [mean_eta],
    'std viscosity [cp]':    [std_eta],
    'sem viscosity [cp]':    [sem_eta]
})

#precompute c = kT / (6*pi*eta) and propagate uncertainty
#uncertainty propagated in quadrature from T and eta (std used as measure of experimental spread; if all files are from one sample, std reflects real instrument uncertainty rather than sample variability)

c = (k_B * mean_T) / (6 * np.pi * mean_eta * 1e-3)  #[m^2/s * Pa*s = m^3 -> m^2/s, converted to SI]

fractional_error_c = np.sqrt(
    (std_T   / mean_T)**2 +
    (std_eta / mean_eta)**2)
delta_c = fractional_error_c * c
relative_error_c = delta_c / c

print(f"\n  c  =  {c:.4e}  ±  {delta_c:.4e}  m³/s")
print(f"  relative uncertainty in c:  {relative_error_c:.3%}")
if relative_error_c > 0.02:
    print(f"  [!] relative uncertainty in c > 2%.")

In [ ]:
#extract scattering intensity (not required for DLS)
perform_intensity_processing = False

if perform_intensity_processing:
    from intensity import extract_intensity
    
    print("Extracting scattering intensity data...")
    
    intensity_data = []
    n_failed = 0
    
    for file in filtered_files:
        extracted_intensity = extract_intensity(file)
        if extracted_intensity is not None:
            filename = os.path.basename(file)
            extracted_intensity['filename'] = filename  
            intensity_data.append(extracted_intensity)
        else:
            n_failed += 1
    
    if not intensity_data:
        raise ValueError("NO intensity data extracted!")
    
    df_intensity = pd.concat(intensity_data, ignore_index=True)
    df_intensity.index = df_intensity.index + 1
    
    print(f"successfully extracted: {len(intensity_data)} files")
    if n_failed > 0:
        print(f"failed to extract: {n_failed} files")
else:
    print("intensity processing disabled")
    df_intensity = None

In [ ]:
if perform_intensity_processing and df_intensity is not None:
    from intensity import plot_meancr
    
    #calculate angle-corrected intensity
    #correction factor: sin(theta) accounts for scattering geometry
    df_intensity['MeanCR_corr [kHz]'] = (
        (df_intensity['meancr0 [kHz]'] + df_intensity['meancr1 [kHz]']) / 2 / 
        (df_intensity['monitordiode [cps]'] * 10**(-3)) * 
        np.sin(np.radians(df_intensity['angle [°]'])))
    
    #plot intensity vs angle
    plot_meancr(df_intensity, 'angle [°]', 'MeanCR_corr [kHz]')
        
    #intensity at 90° (reference angle)
    intensity_90 = df_intensity[df_intensity['angle [°]'] == 90]['MeanCR_corr [kHz]']
    if not intensity_90.empty:
        print(f"intensity at 90°: {intensity_90.mean():.3f} ± {intensity_90.std():.3f} kHz")
    
    #angular range
    print(f"angle range: {df_intensity['angle [°]'].min():.0f}° - {df_intensity['angle [°]'].max():.0f}°")
    print(f"intensity range: {df_intensity['MeanCR_corr [kHz]'].min():.2f} - {df_intensity['MeanCR_corr [kHz]'].max():.2f} kHz")
else:
    pass

In [ ]:
#extracts the time-resolved countrate traces from each file.
#in this setup: 4 detector slots, but slots 1&3 and 2&4 are physically the same detector (cross-correlation mode).
from preprocessing import extract_countrate

all_countrates = {}
n_failed = 0

for file in filtered_files:
    extracted_countrate = extract_countrate(file)
    if extracted_countrate is not None:
        filename = os.path.basename(file)
        all_countrates[filename] = extracted_countrate
    else:
        n_failed += 1
        
    if not all_countrates:
        raise ValueError("NO countrate extracted!")

#rename columns to reflect physical detector layout
column_names = {0: 'time [s]', 
                1: 'detectorslot 1', 
                2: 'detectorslot 2', 
                3: 'detectorslot 3', 
                4: 'detectorslot 4'}
all_countrates = {key: df.rename(columns=column_names) 
                  for key, df in all_countrates.items()}

print(f"Countrate extraction complete: {len(all_countrates)} successful"
      + (f", {n_failed} failed" if n_failed > 0 else ""))

In [ ]:
#optionally plots the full traces for visual quality inspection.

plot_countrate_graphs = False  # set True to inspect traces
allow_data_filtering  = True    # set False to plot only, without exclusion

from preprocessing import plot_countrates, cli_countrate_exclusion

if plot_countrate_graphs:
    if allow_data_filtering:
        filtered_countrates = cli_countrate_exclusion(all_countrates)
        if filtered_countrates and filtered_countrates != all_countrates:
            original_countrates = all_countrates
            all_countrates = filtered_countrates
            n_removed = len(original_countrates) - len(all_countrates)
            print(f"{n_removed} file(s) excluded. "
                  f"{len(all_countrates)} remaining for analysis.")
        else:
            print("NO files excluded. Continuing with full dataset.")
    else:
        plot_countrates(all_countrates, show_indices=False)
else:
    print("countrate plotting skipped.")

In [ ]:
#calculates mean CR per file from the raw countrate traces, applies the sin(theta) geometry correction (but no monitordiode!), and plots the result vs angle.
#if perform_intensity_processing = True, overlays the software-extracted intensity for direct comparison.
import matplotlib.pyplot as plt

cr_rows = []
for filename, df in all_countrates.items():
    mean_cr = (df['detectorslot 1'].mean() + df['detectorslot 2'].mean()) / 2
    cr_rows.append({'filename': filename, 'MeanCR_raw [kHz]': mean_cr})
df_cr = pd.DataFrame(cr_rows)

#merge with basedata to get angles
df_cr = pd.merge(df_cr, 
                 df_basedata_mod[['filename', 'angle [°]']], 
                 on='filename', how='left')
df_cr = df_cr.sort_values('angle [°]').reset_index(drop=True)

#sin(theta) geometry correction
df_cr['MeanCR_corr [kHz]'] = df_cr['MeanCR_raw [kHz]'] * np.sin(np.radians(df_cr['angle [°]']))

#if intensity data available: filter to surviving files, then apply monitordiode normalisation to make both curves directly comparable on the same scale
if perform_intensity_processing and df_intensity is not None:
    df_intensity_plot = df_intensity[df_intensity['filename'].isin(all_countrates.keys())].copy()
    df_cr_merged = pd.merge(df_cr, 
                            df_intensity_plot[['filename', 'monitordiode [cps]']], 
                            on='filename', how='left')
    df_cr['MeanCR_norm [kHz]'] = (df_cr_merged['MeanCR_raw [kHz]'] / 
                                   (df_cr_merged['monitordiode [cps]'] * 1e-3) * 
                                   np.sin(np.radians(df_cr['angle [°]'])))

#plot
fig, ax = plt.subplots(figsize=(8, 4))
if perform_intensity_processing and df_intensity is not None:
    ax.scatter(df_cr['angle [°]'], df_cr['MeanCR_norm [kHz]'],
               alpha=0.7, s=50, label='from countrate traces', color='steelblue')
    ax.scatter(df_intensity_plot.sort_values('angle [°]')['angle [°]'],
               df_intensity_plot.sort_values('angle [°]')['MeanCR_corr [kHz]'],
               alpha=0.7, s=50, label='from intensity extraction', color='tomato')
    ax.legend()
    print("both intensity sources plotted.")
else:
    print("only calculated intensity from extracted countrate plotted.")
    ax.scatter(df_cr['angle [°]'], df_cr['MeanCR_corr [kHz]'],
               alpha=0.7, s=50, color='steelblue')

ax.set_xlabel('angle [°]')
ax.set_ylabel('corrected mean intensity [kHz]')
ax.set_title(experiment_name)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
#extracts g(2)-1 correlation functions for all files that survived countrate filtering.
#4 correlation channels are extracted. In cross-correlation mode, channels 1&2 are the relevant pair. 
#-> channels 3&4 may be zero depending on measurement settings.

from preprocessing import extract_correlation

file_to_path = {os.path.basename(file): file for file in filtered_files}

all_correlations = {}
n_failed = 0
n_missing = 0

for filename in all_countrates.keys():
    if filename in file_to_path:
        extracted_correlation = extract_correlation(file_to_path[filename])
        if extracted_correlation is not None:
            all_correlations[filename] = extracted_correlation
        else:
            n_failed += 1
            print(f"[!]failed to extract correlation from: {filename}")
    else:
        n_missing += 1
        print(f"[!]NO matching file found for: {filename}")

#rename columns
column_names = {0: 'time [ms]', 1: 'correlation 1', 2: 'correlation 2',3: 'correlation 3', 4: 'correlation 4'}
all_correlations = {key: df.rename(columns=column_names)
                    for key, df in all_correlations.items()}

print(f"Correlation extraction complete: {len(all_correlations)} successful"
      + (f", {n_failed} failed" if n_failed > 0 else ""))

In [ ]:
#optionally plots the full correlations for visual quality inspection.

plot_correlation_graphs = False   # set True to inspect curves
allow_correlation_filtering = True  # set False to plot only, without exclusion

from preprocessing import plot_correlations, cli_correlation_exclusion

if plot_correlation_graphs:
    if allow_correlation_filtering:
        filtered_correlations = cli_correlation_exclusion(all_correlations)
        if filtered_correlations and filtered_correlations != all_correlations:
            original_correlations = all_correlations
            all_correlations = filtered_correlations
            n_removed = len(original_correlations) - len(all_correlations)
            print(f"{n_removed} file(s) excluded. "
                  f"{len(all_correlations)} remaining for analysis.")
        else:
            print("NO files excluded. Continuing with full dataset.")
    else:
        plot_correlations(all_correlations, show_indices=False)
else:
    print("correlation plotting skipped.")

In [ ]:
#locks in the final working datasets after all filtering steps.
#df_basedata_mod is re-synced here to exactly match the files that survived both countrate and correlation filtering 
#processed_correlations_1 contains the cross-correlation mean [time in seconds, mean of channels 1 and 2] ready for fitting.

from preprocessing import remove_from_data, remove_dataframes, process_correlation_data

#lock in final correlation dataset
all_correlations_mod = all_correlations.copy()

#re-sync df_basedata_mod to surviving files
files_to_exclude = [f for f in df_basedata_mod['filename']
                    if f not in all_correlations_mod.keys()]

if files_to_exclude:
    df_basedata_mod = remove_from_data(df_basedata_mod, files_to_exclude)
    df_basedata_mod = df_basedata_mod.reset_index(drop=True)
    df_basedata_mod.index = df_basedata_mod.index + 1
    print(f"df_basedata_mod updated: {len(files_to_exclude)} file(s) removed "
          f"to match filtered correlations.")

print(f"final dataset: {len(all_correlations_mod)} correlation(s), "
      f"{len(df_basedata_mod)} basedata entries.")

#process into cross-correlation mean for fitting
#drops raw channel columns and converts time to seconds
columns_to_drop = ['time [ms]', 'correlation 1', 'correlation 2', 'correlation 3', 'correlation 4']
processed_correlations_1 = process_correlation_data(all_correlations_mod, columns_to_drop)

In [ ]:
#OPTIONAL NOISE CORRECTION ON CORRELATION DATA
perform_noise_correction        = False

#baseline_correction : subtracts the mean of the last X% of points from the entire curve. Removes residual DC offset.
baseline_correction  = True
baseline_pct         = 10      # % of tail points used for baseline estimate

#intercept_correction : replaces the first X% of points with their flat mean. 
#stabilises the plateau (beta) used as the fit amplitude in all downstream methods.
intercept_correction = True
intercept_pct        = 10      # % of head points used for plateau estimate

plot_correction      = True   #before/after plot

#column names in processed_correlations_1
col      = 'g(2)-1'
time_col = 't [s]'

from noise import apply_noise_corrections, plot_correction_sample

if perform_noise_correction:
    processed_correlations_2 = apply_noise_corrections(
        processed_correlations_1, col,
        baseline_correction, baseline_pct,
        intercept_correction, intercept_pct
    )
    print(f"Noise correction applied to {len(processed_correlations_2)} files.")
    print(f"  Baseline:  mean of last {baseline_pct}% subtracted"   if baseline_correction  else "  Baseline:  skipped")
    print(f"  Intercept: first {intercept_pct}% flattened to mean"  if intercept_correction else "  Intercept: skipped")

    if plot_correction:
        plot_correction_sample(
            processed_correlations_1, processed_correlations_2,
            df_basedata=df_basedata_mod,
            col=col, time_col=time_col,
            title=experiment_name
        )
else:
    processed_correlations_2 = processed_correlations_1
    print("Noise correction skipped. processed_correlations_2 = processed_correlations_1.")

#### II. CUMULANT METHODS

In [ ]:
#toggle which methods to run
# ============================================================
perform_cumulant_A = False  
perform_cumulant_B = False
perform_cumulant_C = False
perform_cumulant_D = False

##### CUMULANT-METHOD A
{using cumulant-fit data from ALV-Software}

In [ ]:
#extract cumulant data only for files in the filtered correlations
show_raw_data = False    #print extracted cumulant table
export_data   = False    #save cumulant_method_A_data to .txt

if perform_cumulant_A:
    from cumulants import extract_cumulants
    from preprocessing import remove_from_data

    print("Extracting data from files...")

    #map filename -> full path for all filtered files
    file_to_path = {os.path.basename(f): f for f in filtered_files}

    #extract cumulants for all files present in the filtered correlations
    all_data = []
    for filename in all_correlations_mod.keys():
        if filename in file_to_path:
            extracted = extract_cumulants(file_to_path[filename])
            if extracted is not None:
                extracted['filename'] = filename
                all_data.append(extracted)
        else:
            print(f"Warning: no matching file found for {filename}")

    if all_data:
        df_extracted_cumulants = pd.concat(all_data, ignore_index=True)
    else:
        print("No cumulant data extracted!")
        df_extracted_cumulants = pd.DataFrame(columns=[
            'filename',
            '1st order frequency [1/ms]', '2nd order frequency [1/ms]', '3rd order frequency [1/ms]',
            '2nd order frequency exp param [ms^2]', '3rd order frequency exp param [ms^2]'
        ])

    #apply exclusions if already defined upstream
    if 'files_to_exclude' in locals():
        df_extracted_cumulants_mod = remove_from_data(df_extracted_cumulants, files_to_exclude)
    else:
        df_extracted_cumulants_mod = df_extracted_cumulants

    df_extracted_cumulants_mod = df_extracted_cumulants_mod.reset_index(drop=True)
    df_extracted_cumulants_mod.index = df_extracted_cumulants_mod.index + 1

    #merge with basedata
    cumulant_method_A_data = pd.merge(df_basedata_mod, df_extracted_cumulants_mod, on='filename', how='outer')
    cumulant_method_A_data = cumulant_method_A_data.reset_index(drop=True)
    cumulant_method_A_data.index = cumulant_method_A_data.index + 1

    print(f"Cumulant Method A: extracted data for {len(df_extracted_cumulants_mod)} files.")

    if show_raw_data:
        print("\nExtracted cumulant data:")
        print(cumulant_method_A_data.to_string())
        
    if export_data:
        fname = f'Cumulant_Method_A_data_{experiment_name}.txt'
        cumulant_method_A_data.to_csv(fname, sep='\t', index=False)
        print(f"Data exported to {fname}")
else:
    print("Cumulant Method A skipped.")

In [ ]:
#linear regression to determine D
fit_x_range = None   # e.g. (0, 0.001) to restrict fit range, or None for auto

if perform_cumulant_A:
    from cumulants import analyze_diffusion_coefficient

    cumulant_method_A_diff = analyze_diffusion_coefficient(
        data_df      = cumulant_method_A_data,
        q_squared_col= 'q^2',
        gamma_cols   = [
            '1st order frequency [1/ms]',
            '2nd order frequency [1/ms]',
            '3rd order frequency [1/ms]'
        ],
        gamma_unit   = '1/ms',   # ALV data: 1/ms
        x_range      = fit_x_range
    )
else:
    print("Cumulant Method A skipped — regression not executed.")

In [ ]:
#calculate results (D, PDI, Skewness and Rh)
from IPython.display import display
if perform_cumulant_A:
    #polydispersity index from 2nd and 3rd order cumulants
    cumulant_method_A_data['PDI_2nd'] = (
        cumulant_method_A_data['2nd order frequency exp param [ms^2]'] /
        cumulant_method_A_data['2nd order frequency [1/ms]']**2)
    cumulant_method_A_data['PDI_3rd'] = (
        cumulant_method_A_data['3rd order frequency exp param [ms^2]'] /
        cumulant_method_A_data['3rd order frequency [1/ms]']**2)
    polydispersity_method_A_2 = cumulant_method_A_data['PDI_2nd'].mean()
    polydispersity_method_A_3 = cumulant_method_A_data['PDI_3rd'].mean()
    
    #skewness from 3rd order cumulants
    cumulant_method_A_data['Skewness_3rd'] = (
        cumulant_method_A_data['3rd order frequency exp param [ms^2]'] /
        cumulant_method_A_data['2nd order frequency exp param [ms^2]']**(3/2))
    skewness_method_A_3 = cumulant_method_A_data['Skewness_3rd'].mean()
    
    #build results table
    fit_labels = [
        'Rh from 1st order cumulant fit',
        'Rh from 2nd order cumulant fit',
        'Rh from 3rd order cumulant fit',]
    results = []
    for i in range(3):
        D      = cumulant_method_A_diff['q^2_coef'].iloc[i] * 1e-15
        D_err  = cumulant_method_A_diff['q^2_se'].iloc[i]   * 1e-15
        Rh     = c / D * 1e9
        Rh_err = np.sqrt((delta_c / c)**2 + (D_err / D)**2) * Rh
        row = {
            'Fit'           : fit_labels[i],
            'D [m²/s]'      : D,
            'D error [m²/s]': D_err,
            'Rh [nm]'       : Rh,
            'Rh error [nm]' : Rh_err,
            'R_squared'     : cumulant_method_A_diff['R_squared'].iloc[i],
            'Residuals'     : cumulant_method_A_diff['Normality'].iloc[i],
            'PDI'           : polydispersity_method_A_2 if i == 1 else polydispersity_method_A_3 if i == 2 else np.nan,
            'Skewness'      : skewness_method_A_3 if i == 2 else np.nan,
            'Kurtosis'      : np.nan,
        }
        results.append(row)
    method_A_cumulant_result = pd.DataFrame(results)
    display(method_A_cumulant_result)
else:
    method_A_cumulant_result = pd.DataFrame({
        'Fit'           : ['Rh from 1st order cumulant fit',
                           'Rh from 2nd order cumulant fit',
                           'Rh from 3rd order cumulant fit'],
        'D [m²/s]'      : [0, 0, 0],
        'D error [m²/s]': [0, 0, 0],
        'Rh [nm]'       : [0, 0, 0],
        'Rh error [nm]' : [0, 0, 0],
        'R_squared'     : [0, 0, 0],
        'Residuals'     : [0, 0, 0],
        'PDI'           : [np.nan, 0, 0],
        'Skewness'      : [np.nan, np.nan, 0],
        'Kurtosis'      : [np.nan, np.nan, np.nan],
    })
    print("Cumulant Method A skipped — zero result placeholder created.")

##### CUMULANT-METHOD B
{linear regression of ln√(g⁽²⁾−1)}

In [ ]:
#fitting
fit_limits = (0, 0.0002)  #time window [s] for individual curve fitting — keep narrow (only valid for small lag times)
xlim = None #(0, 0.0002) #plotting limits x-axis or None for auto
ylim = None #(0, 0.2) #plotting limits y-axis or None for auto
#fit function: ln(sqrt(g(2)-1)) = 0.5*ln(a) - b*t, wheras b = decay rate Gamma [1/s]

if perform_cumulant_B:
    from cumulants import calculate_g2_B, plot_processed_correlations, remove_rows_by_index

    #compute sqrt(g(2)-1), drop non-positive values
    processed_correlations_B = calculate_g2_B(processed_correlations_2)

    #fit each correlation individually and plot
    cumulant_method_B_fit = plot_processed_correlations(
        processed_correlations_B, fit_limits, xlim, ylim)

    #merge fit results with basedata
    cumulant_method_B_data = pd.merge(
        df_basedata_mod, cumulant_method_B_fit, on='filename', how='outer')
    cumulant_method_B_data = cumulant_method_B_data.reset_index(drop=True)
    cumulant_method_B_data.index = cumulant_method_B_data.index + 1

    print(f"Cumulant Method B: fitted {len(cumulant_method_B_data)} files.")
else:
    print("Cumulant Method B skipped.")

In [ ]:
remove_bad_fits = False   # set True to interactively remove bad fits by index
show_raw_data   = False   # print fit results table
export_data     = False   # save cumulant_method_B_data to .txt

if perform_cumulant_B:
    from cumulants import remove_rows_by_index

    if remove_bad_fits:
        print("Data table (for index reference):")
        print(cumulant_method_B_data[['filename', 'Gamma', 'R_squared']].to_string())
        indices_input = input("\nEnter row indices to remove (comma-separated, or leave empty): ")
        cumulant_method_B_data_mod = remove_rows_by_index(cumulant_method_B_data, indices_input)
    else:
        cumulant_method_B_data_mod = cumulant_method_B_data

    if show_raw_data:
        print("\nFit results:")
        print(cumulant_method_B_data_mod.to_string())

    if export_data:
        fname = f'Cumulant_Method_B_data_{experiment_name}.txt'
        cumulant_method_B_data_mod.to_csv(fname, sep='\t', index=False)
        print(f"Data exported to {fname}")
else:
    print("Cumulant Method B skipped.")

In [ ]:
#linear regression to determine D
fit_x_range = (0, 0.001)   # q^2 fit range [nm^2], or None for auto

if perform_cumulant_B:
    from cumulants import analyze_diffusion_coefficient
    cumulant_method_B_diff = analyze_diffusion_coefficient(
        data_df       = cumulant_method_B_data_mod,
        q_squared_col = 'q^2',
        gamma_cols    = ['Gamma'],
        method_names  = ['Method B'],
        x_range       = fit_x_range
    )
else:
    print("Cumulant Method B skipped — regression not executed.")

In [ ]:
#calculate results (D, PDI and Rh)
from IPython.display import display

if perform_cumulant_B:
    # D [m^2/s]: Gamma in 1/s, q^2 in nm^-2 → *1e-18
    D     = cumulant_method_B_diff['q^2_coef'].iloc[0] * 1e-18
    D_err = cumulant_method_B_diff['q^2_se'].iloc[0]   * 1e-18
    Rh    = c / D * 1e9
    Rh_err = np.sqrt((delta_c / c)**2 + (D_err / D)**2) * Rh

    method_B_cumulant_result = pd.DataFrame([{
        'Fit'           : 'Rh from linear cumulant fit',
        'D [m²/s]'      : D,
        'D error [m²/s]': D_err,
        'Rh [nm]'       : Rh,
        'Rh error [nm]' : Rh_err,
        'R_squared'     : cumulant_method_B_diff['R_squared'].iloc[0],
        'Residuals'     : cumulant_method_B_diff['Normality'].iloc[0],
        'PDI'           : np.nan, 
        'Skewness'      : np.nan,
        'Kurtosis'      : np.nan,
    }])
    display(method_B_cumulant_result)

else:
    method_B_cumulant_result = pd.DataFrame([{
        'Fit'           : 'Rh from linear cumulant fit',
        'D [m²/s]'      : 0, 'D error [m²/s]': 0,
        'Rh [nm]'       : 0, 'Rh error [nm]' : 0,
        'R_squared'     : 0, 'Residuals'     : 0, 
        'PDI'           : np.nan, 'Skewness'      : np.nan,
        'Kurtosis'      : np.nan,
    }])
    print("Cumulant Method B skipped — zero result placeholder created.")

##### CUMULANT-METHOD C
{iterative nonlinear fit}

In [ ]:
#fit functions
def fit_function1(x, a, b, f): #1st order cumulant
    return f + a * np.exp(-2 * b * x)
    
def fit_function2(x, a, b, c, f): #2nd order cumulant
    inner_term = 1 + 0.5 * c * x**2
    return f + a * (np.exp(-b * x) * inner_term)**2

def fit_function3(x, a, b, c, d, f): #3rd order cumulant
    inner_term = 1 + 0.5 * c * x**2 - (d * x**3) / 6
    return f + a * (np.exp(-b * x) * inner_term)**2

def fit_function4(x, a, b, c, d, e, f): #4th order cumulant
    inner_term = 1 + 0.5 * c * x**2 - (d * x**3) / 6 + ((e - 3 * c**2) * x**4) / 24
    return f + a * (np.exp(-b * x) * inner_term)**2

#chosen_fit_functions = [fit_function1, fit_function2, fit_function3, fit_function4]  # compare all orders
chosen_fit_functions = [fit_function2]

#initial parameters [a, b, c, d, e, f]
# a: amplitude ≈ beta (typically 0.8–0.9)
# b: mean decay rate Gamma [1/s] — adjust to your expected size range
# c: 2nd cumulant (≈ 0 monodisperse, > 0 polydisperse)
# d: 3rd cumulant — asymmetry (start at 0)
# e: 4th cumulant (start at 0)
# f: baseline offset (≈ 0 after noise correction)
base_initial_parameters = [0.8, 10000, 0, 0, 0, 0]

#adaptive initial parameter strategy instead of giving initial parameters
adaptive_initial_guesses = True
adaptation_strategy      = 'individual'   #'individual', 'global', or 'representative'

if perform_cumulant_C:
    from cumulants_C import (get_adaptive_initial_parameters, get_meaningful_parameters,
                              plot_processed_correlations_iterative)
    all_fits = []
    plot_offset = 1
    for fit_func in chosen_fit_functions:
        print(f"Fitting with {fit_func.__name__}...")

        if adaptive_initial_guesses:
            initial_parameters = get_adaptive_initial_parameters(
                processed_correlations_2, fit_func, base_initial_parameters,
                strategy=adaptation_strategy, verbose=False
            )
        else:
            initial_parameters = get_meaningful_parameters(fit_func, base_initial_parameters)

        fit_result = plot_processed_correlations_iterative(
            processed_correlations_2, fit_func, (1e-9, 10),
            initial_parameters, method='lm',
            plot_number_start=plot_offset
        )
        fit_result['fit_function'] = fit_func.__name__
        plot_offset += len(fit_result)
        all_fits.append(fit_result)

    #concatenate results from all fit functions — continuous index across all
    cumulant_method_C_fit = pd.concat(all_fits, ignore_index=True)

    #merge with basedata
    cumulant_method_C_data = pd.merge(
        df_basedata_mod, cumulant_method_C_fit, on='filename', how='outer'
    )
    cumulant_method_C_data = cumulant_method_C_data.reset_index(drop=True)
    cumulant_method_C_data.index = cumulant_method_C_data.index + 1

    print(f"\nCumulant Method C: {len(chosen_fit_functions)} function(s), "
          f"{len(cumulant_method_C_data)} total rows.")
else:
    print("Cumulant Method C skipped.")

In [ ]:
remove_bad_fits = False   # set True to interactively remove fits by index
show_raw_data   = False   # print full data table
export_data     = False   # save to .txt

if perform_cumulant_C:
    from cumulants import remove_rows_by_index

    if show_raw_data:
        print(cumulant_method_C_data.to_string())

    if remove_bad_fits:
        #print compact reference table: index, filename, fit function, R^2
        print("Data table (for index reference):")
        print(cumulant_method_C_data[
            ['filename', 'fit_function', 'best_R-squared']
        ].to_string())
        indices_input = input("\nEnter row indices to remove (comma-separated, or leave empty): ")
        cumulant_method_C_data_mod = remove_rows_by_index(cumulant_method_C_data, indices_input)
    else:
        cumulant_method_C_data_mod = cumulant_method_C_data
        print(f"No removal — continuing with {len(cumulant_method_C_data_mod)} rows.")

    if export_data:
        fname = f'Cumulant_Method_C_data_{experiment_name}.txt'
        cumulant_method_C_data_mod.to_csv(fname, sep='\t', index=False)
        print(f"Data exported to {fname}")

else:
    print("Cumulant Method C skipped.")

In [ ]:
#linear regression to determine D
fit_x_range = None   # e.g. (0, 0.001) or None for auto

if perform_cumulant_C:
    from cumulants import analyze_diffusion_coefficient

    diff_results = []
    for fit_func in chosen_fit_functions:
        subset = cumulant_method_C_data_mod[
            cumulant_method_C_data_mod['fit_function'] == fit_func.__name__
        ].copy()

        result = analyze_diffusion_coefficient(
            data_df       = subset,
            q_squared_col = 'q^2',
            gamma_cols    = ['best_b'],
            method_names  = [fit_func.__name__],
            x_range       = fit_x_range
        )
        diff_results.append(result)

    cumulant_method_C_diff = pd.concat(diff_results, ignore_index=True)
else:
    print("Cumulant Method C skipped — regression not executed.")

In [ ]:
#calculate results (D, PDI and Rh)
from IPython.display import display
fit_labels = {
    'fit_function1': '1st order cumulant fit',
    'fit_function2': '2nd order cumulant fit',
    'fit_function3': '3rd order cumulant fit',
    'fit_function4': '4th order cumulant fit',}

if perform_cumulant_C:
    results = []
    for i, fit_func in enumerate(chosen_fit_functions):
        subset = cumulant_method_C_data_mod[
            cumulant_method_C_data_mod['fit_function'] == fit_func.__name__
        ].copy()

        # PDI (not defined for 1st order)
        if fit_func.__name__ == 'fit_function1':
            pdi = np.nan
        else:
            pdi = (subset['best_c'] / subset['best_b']**2).mean()
        # Skewness (only for fit_function3 and fit_function4)
        if fit_func.__name__ in ('fit_function3', 'fit_function4'):
            skewness = (subset['best_d'] / subset['best_c']**(3/2)).mean()
        else:
            skewness = np.nan
        # Kurtosis (only for fit_function4)
        if fit_func.__name__ == 'fit_function4':
            kurtosis = (subset['best_e'] / subset['best_c']**2).mean()
        else:
            kurtosis = np.nan

        D     = cumulant_method_C_diff['q^2_coef'].iloc[i] * 1e-18
        D_err = cumulant_method_C_diff['q^2_se'].iloc[i]   * 1e-18
        Rh    = c / D * 1e9
        Rh_err = np.sqrt((delta_c / c)**2 + (D_err / D)**2) * Rh

        results.append({
            'Fit'           : f'Rh from {fit_labels.get(fit_func.__name__, fit_func.__name__)}',
            'D [m²/s]'      : D,
            'D error [m²/s]': D_err,
            'Rh [nm]'       : Rh,
            'Rh error [nm]' : Rh_err,
            'R_squared'     : cumulant_method_C_diff['R_squared'].iloc[i],
            'Residuals'     : cumulant_method_C_diff['Normality'].iloc[i],
            'PDI'           : pdi,
            'Skewness'      : skewness,
        })

    method_C_cumulant_result = pd.DataFrame(results)
    display(method_C_cumulant_result)

else:
    all_fit_labels = {
        'fit_function1': 'Rh from 1st order cumulant fit',
        'fit_function2': 'Rh from 2nd order cumulant fit',
        'fit_function3': 'Rh from 3rd order cumulant fit',
        'fit_function4': 'Rh from 4th order cumulant fit',
    }
    method_C_cumulant_result = pd.DataFrame([{
        'Fit'           : label,
        'D [m²/s]'      : 0,
        'D error [m²/s]': 0,
        'Rh [nm]'       : 0,
        'Rh error [nm]' : 0,
        'R_squared'     : 0,
        'Residuals'     : 0,
        'PDI'           : np.nan if name == 'fit_function1' else 0,
        'Skewness'      : np.nan if name in ('fit_function1', 'fit_function2') else 0,
        'Kurtosis'      : np.nan if name != 'fit_function4' else 0,
    } for name, label in all_fit_labels.items()])
    print("Cumulant Method C skipped — zero result placeholder created.")

##### CUMULANT-METHOD D
{multimodal cumulant analysis via dirac delta modes}

In [ ]:
#fitting
n_max         = 25 #maximum number of Dirac delta modes to try
n_start       = 1 #starting number of modes (1 for monomodal, 3-5 for known polydisperse)
gap_threshold = 3.0 #minimum log-ratio between adjacent gammas to count as separate population

if perform_cumulant_D:
    from cumulants_D import fit_correlations_method_D
    from cumulants import remove_rows_by_index

    cumulant_method_D_fit = fit_correlations_method_D(
        processed_correlations_2,
        n_max=n_max, n_start=n_start,
        gap_threshold=gap_threshold,
        plot=True
    )

    #merge with basedata
    cumulant_method_D_data = pd.merge(
        df_basedata_mod, cumulant_method_D_fit, on='filename', how='outer'
    )
    cumulant_method_D_data = cumulant_method_D_data.reset_index(drop=True)
    cumulant_method_D_data.index = cumulant_method_D_data.index + 1

    print(f"\nCumulant Method D: fitted {len(cumulant_method_D_data)} files.")
    print(f"Populations found per file: {cumulant_method_D_data['n_populations'].value_counts().sort_index().to_dict()}")
else:
    print("Cumulant Method D skipped.")

In [ ]:
remove_bad_fits = False   # set True to interactively remove bad fits by index
show_raw_data   = False   # print full data table
export_data     = False   # save to .txt

if perform_cumulant_D:
    from cumulants import remove_rows_by_index

    if show_raw_data:
        print(cumulant_method_D_data.to_string())

    if remove_bad_fits:
        print("Data table (for index reference):")
        print(cumulant_method_D_data[
            ['filename', 'n_populations', 'pdi', 'R-squared']
        ].to_string())
        indices_input = input("\nEnter row indices to remove (comma-separated, or leave empty): ")
        cumulant_method_D_data_mod = remove_rows_by_index(cumulant_method_D_data, indices_input)
    else:
        cumulant_method_D_data_mod = cumulant_method_D_data

    if export_data:
        fname = f'Cumulant_Method_D_data_{experiment_name}.txt'
        cumulant_method_D_data_mod.to_csv(fname, sep='\t', index=False)
        print(f"Data exported to {fname}")
else:
    print("Cumulant Method D skipped.")

In [ ]:
#clustering
enable_clustering     = True    # False = pass-through, use gamma columns as-is
normalize_by_q2       = True    # cluster on D = Γ/q^2 instead of Γ (recommended)
n_clusters            = 'auto'  # or set integer
distance_threshold    = 2.0     # log₁₀ space — adjust 0.3–2.0
min_abundance         = 0.3     # minimum fraction of files for a reliable population
clustering_strategy   = 'silhouette_refined'  # 'simple' or 'silhouette_refined'
uncertainty_flags     = False # True to flag and possible remove unreliable data
uncertainty_threshold = 0.5

if perform_cumulant_D:
    from clustering import cluster_all_gammas, get_reliable_gamma_cols

    gamma_cols = [col for col in cumulant_method_D_data_mod.columns
                  if col.startswith('gamma_') and col != 'gamma_mean']

    cumulant_method_D_clustered, cluster_info = cluster_all_gammas(
        cumulant_method_D_data_mod,
        gamma_cols            = gamma_cols,
        q_squared_col         = 'q^2',
        enable_clustering     = enable_clustering,
        normalize_by_q2       = normalize_by_q2,
        n_clusters            = n_clusters,
        distance_threshold    = distance_threshold,
        min_abundance         = min_abundance,
        clustering_strategy   = clustering_strategy,
        uncertainty_flags     = uncertainty_flags,
        uncertainty_threshold = uncertainty_threshold,
        plot=True
    )

    reliable_gamma_cols = get_reliable_gamma_cols(cluster_info)
    print(f"\n{cluster_info['n_populations']} reliable population(s) found.")
else:
    print("Cumulant Method D skipped — clustering not executed.")

In [ ]:
#linear regression
fit_x_range = None   # e.g. (0, 0.001) or None for auto

if perform_cumulant_D:
    from cumulants import analyze_diffusion_coefficient

    cumulant_method_D_diff = analyze_diffusion_coefficient(
        data_df       = cumulant_method_D_clustered,
        q_squared_col = 'q^2',
        gamma_cols    = reliable_gamma_cols,
        x_range       = fit_x_range
    )
else:
    print("Cumulant Method D skipped — regression not executed.")

In [ ]:
#calculate results (D, PDI and Rh)
from IPython.display import display

if perform_cumulant_D:
    results = []
    for idx, row in cumulant_method_D_diff.iterrows():
        pop_num = idx + 1
        D     = row['q^2_coef'] * 1e-18
        D_err = row['q^2_se']   * 1e-18
        Rh    = c / D * 1e9
        Rh_err = np.sqrt((delta_c / c)**2 + (D_err / D)**2) * Rh

        result = {
            'Fit'            : f'Population {pop_num}',
            'D [m²/s]'       : D,
            'D error [m²/s]' : D_err,
            'Rh [nm]'        : Rh,
            'Rh error [nm]'  : Rh_err,
            'R_squared'      : row['R_squared'],
            'Residuals'      : row['Normality'],
            'PDI'            : cumulant_method_D_data_mod['pdi'].mean(),
            'Skewness'       : cumulant_method_D_data_mod['skewness'].mean(),
            'Kurtosis'       : cumulant_method_D_data_mod['kurtosis'].mean(),
        }
        if idx < len(cluster_info['population_abundances']):
            result['Abundance'] = f"{cluster_info['population_abundances'][idx]*100:.1f}%"
        if cluster_info['silhouette_score'] is not None:
            result['Silhouette'] = cluster_info['silhouette_score']

        results.append(result)

    method_D_cumulant_result = pd.DataFrame(results)
    display(method_D_cumulant_result)

else:
    method_D_cumulant_result = pd.DataFrame([{
        'Fit'           : 'Population 1',
        'D [m²/s]'      : 0, 'D error [m²/s]': 0,
        'Rh [nm]'       : 0, 'Rh error [nm]' : 0,
        'R_squared'     : 0, 'Residuals'      : 0,
        'PDI'           : 0, 'Skewness'       : 0,
        'Kurtosis'      : 0,
    }])
    print("Cumulant Method D skipped — zero result placeholder created.")

#### III. INVERSE-LAPLACIAN METHODS

In [ ]:
#toggle which methods to run
# ============================================================
perform_nnls = True
analyze_alpha = False
regularized_fit = True

##### NNLS
{non-negative least squares fit}

In [ ]:
#simple NNLS-Fit without additional constraints
decay_times = np.logspace(-8, 1, 100)  # lag time grid for inverse Laplace — 100-400 points recommended
prominence  = 0.05   # peak detection sensitivity — lower = more sensitive
distance    = 1      # minimum distance between detected peaks

nnls_params = {
    'decay_times': decay_times,
    'prominence' : prominence,
    'distance'   : distance,}

if perform_nnls:
    from regularized import nnls_all

    nnls_fit = nnls_all(processed_correlations_2, nnls_params)

    #merge with basedata
    nnls_data = pd.merge(df_basedata_mod, nnls_fit, on='filename', how='outer')
    nnls_data = nnls_data.reset_index(drop=True)
    nnls_data.index = nnls_data.index + 1

    print(f"\nNNLS: fitted {len(nnls_data)} files.")
else:
    print("NNLS skipped.")

In [ ]:
remove_bad_fits = False   # set True to interactively remove bad fits by index
show_raw_data   = False   # print full data table
export_data     = False   # save to .txt

if perform_nnls:
    from cumulants import remove_rows_by_index

    if show_raw_data:
        print(nnls_data.to_string())

    if remove_bad_fits:
        print("Data table (for index reference):")
        print(nnls_data[['filename', 'R_squared']].to_string())
        indices_input = input("\nEnter row indices to remove (comma-separated, or leave empty): ")
        nnls_data_mod = remove_rows_by_index(nnls_data, indices_input)
    else:
        nnls_data_mod = nnls_data

    if export_data:
        fname = f'NNLS_data_{experiment_name}.txt'
        nnls_data_mod.to_csv(fname, sep='\t', index=False)
        print(f"Data exported to {fname}")
else:
    print("NNLS skipped.")

In [ ]:
#calculating decay rates from decay times
if perform_nnls:
    from regularized import calculate_decay_rates

    # Auto-detect tau columns
    tau_cols = sorted([col for col in nnls_data_mod.columns if col.startswith('tau_')])
    print(f"Detected tau columns: {tau_cols}")

    nnls_data_mod = calculate_decay_rates(nnls_data_mod, tau_cols)
    print(f"Computed gamma columns: {[col.replace('tau', 'gamma') for col in tau_cols]}")
else:
    print("NNLS skipped.")

In [ ]:
#clustering
enable_clustering     = True
normalize_by_q2       = True
n_clusters            = 'auto'
distance_threshold    = 2.0
min_abundance         = 0.3
clustering_strategy   = 'silhouette_refined'   # 'simple' or 'silhouette_refined'
uncertainty_flags     = False
uncertainty_threshold = 0.5

if perform_nnls:
    from clustering import cluster_all_gammas, get_reliable_gamma_cols

    gamma_cols = sorted([col for col in nnls_data_mod.columns if col.startswith('gamma_')])

    nnls_data_clustered, cluster_info = cluster_all_gammas(
        nnls_data_mod,
        gamma_cols            = gamma_cols,
        q_squared_col         = 'q^2',
        enable_clustering     = enable_clustering,
        normalize_by_q2       = normalize_by_q2,
        n_clusters            = n_clusters,
        distance_threshold    = distance_threshold,
        min_abundance         = min_abundance,
        clustering_strategy   = clustering_strategy,
        uncertainty_flags     = uncertainty_flags,
        uncertainty_threshold = uncertainty_threshold,
        plot=True
    )

    reliable_gamma_cols = get_reliable_gamma_cols(cluster_info)
    print(f"\n{cluster_info['n_populations']} reliable population(s) found.")
else:
    print("NNLS skipped — clustering not executed.")

In [ ]:
#linear regression
fit_x_range = None   # e.g. (0, 0.001) or None for auto

if perform_nnls:
    from cumulants import analyze_diffusion_coefficient

    nnls_diff = analyze_diffusion_coefficient(
        data_df       = nnls_data_clustered,
        q_squared_col = 'q^2',
        gamma_cols    = reliable_gamma_cols,
        x_range       = fit_x_range
    )
else:
    print("NNLS skipped — regression not executed.")

In [ ]:
#calculate results (D, Rh)
from IPython.display import display

if perform_nnls:
    results = []
    for idx, row in nnls_diff.iterrows():
        pop_num = idx + 1
        D      = row['q^2_coef'] * 1e-18
        D_err  = row['q^2_se']   * 1e-18
        Rh     = c / D * 1e9
        Rh_err = np.sqrt((delta_c / c)**2 + (D_err / D)**2) * Rh

        result = {
            'Fit'           : f'Population {pop_num}',
            'D [m²/s]'      : D,
            'D error [m²/s]': D_err,
            'Rh [nm]'       : Rh,
            'Rh error [nm]' : Rh_err,
            'R_squared'     : row['R_squared'],
            'Residuals'     : row['Normality'],
            'PDI'           : np.nan,
            'Skewness'      : np.nan,
            'Kurtosis'      : np.nan,
            'Silhouette'    : cluster_info['silhouette_score'] if cluster_info['silhouette_score'] is not None else np.nan,
        }
        if idx < len(cluster_info['population_abundances']):
            result['Abundance'] = f"{cluster_info['population_abundances'][idx]*100:.1f}%"

        results.append(result)

    nnls_result = pd.DataFrame(results)
    display(nnls_result)

else:
    nnls_result = pd.DataFrame([{
        'Fit'           : 'Population 1',
        'D [m²/s]'      : 0, 'D error [m²/s]': 0,
        'Rh [nm]'       : 0, 'Rh error [nm]' : 0,
        'R_squared'     : 0, 'Residuals'      : 0,
        'PDI'           : np.nan, 'Skewness'  : np.nan,
        'Kurtosis'      : np.nan,
    }])
    print("NNLS skipped — zero result placeholder created.")

##### ALPHA ANALYSIS
{finding suitable and physical meaningful regularization parameter for Regularized Fit}

In [ ]:
#optional, run before regularized fit
#plots randomly selected datasets across a range of alpha values to help determine a suitable smoothing parameter for reg. fit
num_datasets  = 3             # number of randomly selected datasets to plot
alpha_range   = (0.01, 1.0)   # meaningful alpha-range: 0.01–1.0
num_alphas    = 5             # number of alpha values to test across range

alpha_params = {
    'decay_times': np.logspace(-8, 1, 200),
    'prominence' : 0.001,
    'distance'   : 1,
}

if analyze_alpha:
    from regularized import nnls_reg_simple, analyze_random_datasets_grid
    fig, selected_datasets = analyze_random_datasets_grid(
        processed_correlations_2,
        num_datasets             = num_datasets,
        base_nnls_params         = alpha_params,
        nnls_reg_simple_function = nnls_reg_simple,
        alpha_range              = alpha_range,
        num_alphas               = num_alphas
    )
else:
    print("Alpha analysis skipped.")

##### REGULARIZED FIT
{non-negative least squares fit with Tikhonov Regularization}

In [ ]:
#regularized fitting
decay_times        = np.logspace(-8, 1, 200)  # 100-400 points recommended
prominence         = 0.01    # peak detection sensitivity — lower = more sensitive
distance           = 1       # minimum distance between detected peaks
alpha              = 0.2     # smoothing parameter — higher = smoother
normalize          = True    # normalize distribution
peak_method        = 'centroid' # 'maximum' or 'centroid'
sparsity_penalty   = 0       # additional sparsity regularization (0 = disabled)

nnls_reg_params = {
    'decay_times'        : decay_times,
    'prominence'         : prominence,
    'distance'           : distance,
    'alpha'              : alpha,
    'normalize'          : normalize,
    'peak_method'        : peak_method,
    'sparsity_penalty'   : sparsity_penalty,
}

if regularized_fit:
    from regularized import nnls_reg_all
    nnls_reg_df, full_results = nnls_reg_all(processed_correlations_2, nnls_reg_params)

    #merge with basedata
    nnls_reg_data = pd.merge(df_basedata_mod, nnls_reg_df, on='filename', how='outer')
    nnls_reg_data = nnls_reg_data.reset_index(drop=True)
    nnls_reg_data.index = nnls_reg_data.index + 1

    print(f"\nRegularized fit: fitted {len(nnls_reg_data)} files.")
else:
    print("Regularized fit skipped.")

In [ ]:
remove_bad_fits = False   # set True to interactively remove bad fits by index
show_raw_data   = False   # print full data table
export_data     = False   # save to .txt

if regularized_fit:
    from cumulants import remove_rows_by_index

    if show_raw_data:
        print(nnls_reg_data.to_string())

    if remove_bad_fits:
        print("Data table (for index reference):")
        print(nnls_reg_data[['filename', 'R_squared']].to_string())
        indices_input = input("\nEnter row indices to remove (comma-separated, or leave empty): ")
        nnls_reg_data_mod = remove_rows_by_index(nnls_reg_data, indices_input)
    else:
        nnls_reg_data_mod = nnls_reg_data

    if export_data:
        fname = f'Regularized_data_{experiment_name}.txt'
        nnls_reg_data_mod.to_csv(fname, sep='\t', index=False)
        print(f"Data exported to {fname}")
else:
    print("Regularized fit skipped.")

In [ ]:
#calculating decay rates from decay times
if regularized_fit:
    from regularized import calculate_decay_rates

    #auto-detect tau columns
    tau_cols = sorted([col for col in nnls_reg_data_mod.columns if col.startswith('tau_')])
    print(f"Detected tau columns: {tau_cols}")

    nnls_reg_data_mod = calculate_decay_rates(nnls_reg_data_mod, tau_cols)
    print(f"Computed gamma columns: {[col.replace('tau', 'gamma') for col in tau_cols]}")
else:
    print("Regularized fit skipped.")

In [ ]:
#clustering
enable_clustering     = True    # False = pass-through, use gamma columns as-is
normalize_by_q2       = True    # cluster on D = Γ/q^2 instead of Γ (recommended)
n_clusters            = 'auto'  # or set integer
distance_threshold    = 0.5     # log₁₀ space — adjust 0.3–2.0
min_abundance         = 0.3     # minimum fraction of files for a reliable population
clustering_strategy   = 'silhouette_refined'  # 'simple' or 'silhouette_refined'
uncertainty_flags     = True # True to flag and possible remove unreliable data
uncertainty_threshold = 0.5

if regularized_fit:
    from clustering import cluster_all_gammas, get_reliable_gamma_cols

    gamma_cols = sorted([col for col in nnls_reg_data_mod.columns
                         if col.startswith('gamma_')])

    nnls_reg_data_clustered, cluster_info = cluster_all_gammas(
        nnls_reg_data_mod,
        gamma_cols            = gamma_cols,
        q_squared_col         = 'q^2',
        enable_clustering     = enable_clustering,
        normalize_by_q2       = normalize_by_q2,
        n_clusters            = n_clusters,
        distance_threshold    = distance_threshold,
        min_abundance         = min_abundance,
        clustering_strategy   = clustering_strategy,
        uncertainty_flags     = uncertainty_flags,
        uncertainty_threshold = uncertainty_threshold,
        plot=True)

    #per-population skewness/kurtosis averaged across files — τ-space, flip sign to compare with cumulants
    from clustering import aggregate_peak_stats
    cluster_info = aggregate_peak_stats(cluster_info, nnls_reg_data_mod)
    
    reliable_gamma_cols = get_reliable_gamma_cols(cluster_info)
    print(f"\n{cluster_info['n_populations']} reliable population(s) found.")
else:
    print("Regularized fit skipped — clustering not executed.")

In [ ]:
#linear regression
fit_x_range = None   # e.g. (0, 0.001) or None for auto

if regularized_fit:
    from cumulants import analyze_diffusion_coefficient

    nnls_reg_diff = analyze_diffusion_coefficient(
        data_df       = nnls_reg_data_clustered,
        q_squared_col = 'q^2',
        gamma_cols    = reliable_gamma_cols,
        x_range       = fit_x_range
    )
else:
    print("Regularized fit skipped — regression not executed.")

In [ ]:
#calculate results (D,Rh)
from IPython.display import display

if regularized_fit:
    results = []
    for idx, row in nnls_reg_diff.iterrows():
        pop_num = idx + 1
        D      = row['q^2_coef'] * 1e-18
        D_err  = row['q^2_se']   * 1e-18
        Rh     = c / D * 1e9
        Rh_err = np.sqrt((delta_c / c)**2 + (D_err / D)**2) * Rh

        result = {
            'Fit'           : f'Population {pop_num}',
            'D [m²/s]'      : D,
            'D error [m²/s]': D_err,
            'Rh [nm]'       : Rh,
            'Rh error [nm]' : Rh_err,
            'R_squared'     : row['R_squared'],
            'Residuals'     : row['Normality'],
            'PDI'           : np.nan,
            'Skewness'      : cluster_info.get('population_skewness_mean', {}).get(pop_num, np.nan),
            'Kurtosis'      : cluster_info.get('population_kurtosis_mean', {}).get(pop_num, np.nan),
            'Silhouette'    : cluster_info['silhouette_score'] if cluster_info['silhouette_score'] is not None else np.nan,}
            # Note: Skewness and Kurtosis are computed in tau-space (weighted moments of the decay time distribution).
            # Skewness > 0 → tail toward larger tau → larger particles.
            # To compare with cumulant skewness (Gamma-space): flip the sign.
        
        if idx < len(cluster_info['population_abundances']):
            result['Abundance'] = f"{cluster_info['population_abundances'][idx]*100:.1f}%"

        results.append(result)

    nnls_reg_result = pd.DataFrame(results)
    display(nnls_reg_result)

else:
    nnls_reg_result = pd.DataFrame([{
        'Fit'           : 'Population 1',
        'D [m²/s]'      : 0, 'D error [m²/s]': 0,
        'Rh [nm]'       : 0, 'Rh error [nm]' : 0,
        'R_squared'     : 0, 'Residuals'      : 0,
        'PDI'           : np.nan, 'Skewness'  : 0,
        'Kurtosis'      : 0,
    }])
    print("Regularized fit skipped — zero result placeholder created.")

In [ ]:
#possible distribution plots for regularized fit
plot_reg_fit_distr = False

angles          = [50,90,120]      # angles to plot [°]
measurement_mode= 'average'       # 'first', 'average', or 'all'
convert_to_radius = True          # plot Rh [nm] instead of decay time τ
figsize         = (8, 6)
xlim            = (10, 1000)      # adjust range of x-axis !pay attention if you plot tau or Rh!
title           = ''

# set filenames to override angle/measurement_mode selection:
filenames = None
# filenames = ['1_80-30nm_15-200nm_0011_0002.ASC',
#              '1_80-30nm_15-200nm_0012_0002.ASC']

if plot_reg_fit_distr:
    if regularized_fit:
        from regularized import plot_distributions
        plot_distributions(
            full_results, nnls_reg_params, nnls_reg_data,
            angles            = angles,
            measurement_mode  = measurement_mode,
            convert_to_radius = convert_to_radius,
            figsize           = figsize,
            xlim              = xlim,
            title             = title,
            filenames         = filenames)
    else:
        print("Regularized fit skipped — distribution plot not available.")
else:
    print("Plotting of distribution functions disabled.")

In [ ]:
#intensity & guinier analysis (perform_intensity_processing and regularized_fit have to be set True)
perform_sls             = True
log_y                   = True

guinier_q2_range        = None   # None / (q2_min, q2_max) / {1:(a,b), 2:(c,d)}
guinier_q2_range_total  = None

perform_guinier         = True
perform_number_weighted = True
exponent                = 6      #e.g. 6 = compact spheres, 5 = Daoud-Cotton star polymers

if perform_sls and regularized_fit and perform_intensity_processing:
    from sls_functions_for_regularized import (compute_sls_data, compute_sls_data_number_weighted, compute_guinier_total, compute_guinier_extrapolation,
        plot_sls_intensity, plot_guinier, summarize_sls_combined)

    n_populations_sls = cluster_info['n_populations']

    sls_data = compute_sls_data(nnls_reg_data_clustered, df_intensity, n_populations_sls)

    #extract Rh from nnls_reg_result
    rh_values = {}
    for idx, row in nnls_reg_result.iterrows():
        rh_values[idx + 1] = row['Rh [nm]']

    if perform_number_weighted:
        sls_data = compute_sls_data_number_weighted(sls_data, n_populations_sls, rh_values, exponent=exponent)

    plot_sls_intensity(sls_data, n_populations_sls, experiment_name, log_y=log_y)

    total_result = compute_guinier_total(sls_data, q2_range=guinier_q2_range_total)

    if perform_guinier:
        guinier_results = compute_guinier_extrapolation(sls_data, n_populations_sls, q2_range=guinier_q2_range)
        plot_guinier(guinier_results, experiment_name, total_result=total_result)
        display(summarize_sls_combined(sls_data, guinier_results, total_result, n_populations_sls, rh_values, exponent=exponent))

else:
    print("SLS analysis skipped.")

#### IV. RESULTS OVERVIEW

In [ ]:
#COLUMN DESCRIPTIONS
  #Method        Analysis method used (A, B, C = single-mode; D, NNLS, Regularized = multi-mode)
  #Fit           Specific fit variant (e.g. cumulant order, population number)

  #Rh [nm]       Hydrodynamic radius — from Stokes-Einstein: Rh = kT / (6πη·D)
  #Rh error [nm] Propagated uncertainty from D and kT/6πη (T+η)

  #D [m²/s]      Translational diffusion coefficient — slope of Γ vs q² regression
  #D error [m²/s]Standard error of the slope from OLS regression

  #R_squared     Goodness of fit of the Γ vs q² linear regression
  #Residuals     Normality assessment of regression residuals (based on Jarque-Bera + Omnibus tests)

  #PDI           Polydispersity index = µ₂/Γ² — measure of size distribution width
  #                < 0.05: highly monodisperse
  #               0.05–0.20: moderately monodisperse
  #                > 0.30: broad — most likely multimodal

  #Skewness      skewness = µ₃/µ₂^(3/2) 
  #              For Cumulant Methods: Asymmetry of total gamma distribution; NOT per population!; Reg. Fit opposite sign!
  #              For Reg. Fit: Asymmetry of the decay-time distribution (τ-space) [≈ 0: symmetric,  > 0: tail toward larger particles]

  #Kurtosis      kurtosis = µ4/µ₂^2
  #              For Cumulant Methods: Kurtosis of total gamma distribution; NOT per population!; Reg. Fit opposite sign!
  #              For Reg. Fit: Kurtosis of the decay-time distribution peak shape [> 0: sharp/narrow peak,  < 0: broad/flat peak]

  #Abundance     Fraction of files in which this population was detected [0–1]
  #                e.g. 0.85 = present in 85% of measurements across all angles

  #Silhouette    Clustering quality score [−1 to 1]
  #                > 0.7: well-separated populations
  #                0.5–0.7: reasonable separation
  #                < 0.5: overlapping populations — interpret with care

In [ ]:
#SINGLE-MODE COMPARISON TABLE
from IPython.display import display

#add method label column to each result before stacking
def _tag(df, method):
    out = df.copy()
    out.insert(0, 'Method', method)
    return out

single_mode_frames = []
if perform_cumulant_A:
    single_mode_frames.append(_tag(method_A_cumulant_result, 'A'))
if perform_cumulant_B:
    single_mode_frames.append(_tag(method_B_cumulant_result, 'B'))
if perform_cumulant_C:
    single_mode_frames.append(_tag(method_C_cumulant_result, 'C'))

if single_mode_frames:
    df_single_mode_results = pd.concat(single_mode_frames, ignore_index=True)
    display(df_single_mode_results)
else:
    df_single_mode_results = pd.DataFrame()
    print("No single-mode results available.")

In [ ]:
#MULTI-MODE COMPARISON TABLE
from IPython.display import display

multi_mode_frames = []
if perform_cumulant_D:
    multi_mode_frames.append(_tag(method_D_cumulant_result, 'D'))
if perform_nnls:
    multi_mode_frames.append(_tag(nnls_result, 'NNLS'))
if regularized_fit:
    multi_mode_frames.append(_tag(nnls_reg_result, 'Regularized'))

if multi_mode_frames:
    df_multi_mode_results = pd.concat(multi_mode_frames, ignore_index=True)
    display(df_multi_mode_results)
else:
    df_multi_mode_results = pd.DataFrame()
    print("No multi-mode results available.")

In [ ]:
#Rh COMPARISON PLOT
import matplotlib.pyplot as plt

Rh_wmean = Rh_wmean if 'Rh_wmean' in dir() else np.nan
Rh_wstd  = Rh_wstd  if 'Rh_wstd'  in dir() else np.nan

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Single-mode ---
ax = axes[0]
if not df_single_mode_results.empty:
    sm = df_single_mode_results[df_single_mode_results['Rh [nm]'] > 0].copy().reset_index(drop=True)
    colors  = {'A': '#2196F3', 'B': '#4CAF50', 'C': '#FF9800'}
    markers = {'A': 'o',       'B': 's',        'C': '^'}
    seen_methods = set()
    for i, row in sm.iterrows():
        method = row['Method']
        ax.errorbar(
            i, row['Rh [nm]'], yerr=row['Rh error [nm]'],
            fmt=markers.get(method, 'o'),
            color=colors.get(method, 'gray'),
            capsize=4, markersize=7,
            label=method if method not in seen_methods else ''
        )
        seen_methods.add(method)
    ax.set_xticks(range(len(sm)))
    ax.set_xticklabels(sm['Fit'], rotation=30, ha='right', fontsize=8)
    if not np.isnan(Rh_wmean):
        ax.axhline(Rh_wmean, color='black', linestyle='--', linewidth=1,
                   label=f'Weighted mean ({Rh_wmean:.1f} nm)')
        ax.fill_between(range(len(sm)), Rh_wmean - Rh_wstd, Rh_wmean + Rh_wstd,
                        alpha=0.1, color='black')
ax.set_ylabel('Rh [nm]')
ax.set_title('Single-mode methods (A, B, C)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# --- Multi-mode ---
ax = axes[1]
if not df_multi_mode_results.empty:
    mm = df_multi_mode_results[df_multi_mode_results['Rh [nm]'] > 0].copy().reset_index(drop=True)
    colors  = {'D': '#9C27B0', 'NNLS': '#F44336', 'Regularized': '#009688'}
    markers = {'D': 'D',       'NNLS': 'v',        'Regularized': 'P'}
    seen_methods = set()
    for i, row in mm.iterrows():
        method = row['Method']
        ax.errorbar(
            i, row['Rh [nm]'], yerr=row['Rh error [nm]'],
            fmt=markers.get(method, 'o'),
            color=colors.get(method, 'gray'),
            capsize=4, markersize=7,
            label=method if method not in seen_methods else ''
        )
        seen_methods.add(method)
    ax.set_xticks(range(len(mm)))
    ax.set_xticklabels(mm['Fit'], rotation=30, ha='right', fontsize=8)
ax.set_ylabel('Rh [nm]')
ax.set_title('Multi-mode methods (D, NNLS, Regularized)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.suptitle(f'{experiment_name} — Rh comparison', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
#export results
export_single_mode = False
export_multi_mode  = False

if export_single_mode and not df_single_mode_results.empty:
    fname = f'Results_single_mode_{experiment_name}.txt'
    df_single_mode_results.to_csv(fname, sep='\t', index=False)
    print(f"Single-mode results exported to {fname}")

if export_multi_mode and not df_multi_mode_results.empty:
    fname = f'Results_multi_mode_{experiment_name}.txt'
    df_multi_mode_results.to_csv(fname, sep='\t', index=False)
    print(f"Multi-mode results exported to {fname}")

if not export_single_mode and not export_multi_mode:
    print("Export skipped.")